In [29]:
import sys

sys.path.append('/path/to/equinet')

In [30]:
import pandas as pd
import numpy as np

from equinet.train import make_predictions, load_model
from equinet.args import PredictArgs

In [31]:
model_path = "./model.pt"

### Inputting the Whole CLI via Python functions
Predict with full CLI arguments. Output is saved to file and is the return of the function.

In [ ]:
arguments = [
    '--test_path', 'example_test.csv',
    '--features_path', 'example_features.csv',
    '--preds_path', 'example_output.csv',
    '--checkpoint_path', model_path,
    '--number_of_molecules', '2',
    '--num_workers', '0',
]

args = PredictArgs().parse_args(arguments)

preds = make_predictions(args=args)

In [ ]:
print(preds)

### Preloading a model and then running a single line prediction

In [34]:
# If you want a temporary path location for the results to write to
import tempfile

temp_preds = tempfile.mktemp(suffix='.csv')
temp_features = tempfile.mktemp(suffix='.csv')
temp_test = tempfile.mktemp(suffix='.csv')

In [35]:
# Enter in the input data that you want to predict on.
# For EquiNet we need: [Smiles_1, Smiles_2, x1, x2, T] and dummy inputs for [log10P1sat, log10P2sat]
# make a list for each

smiles_1 = ['CCO', 'CCN']
smiles_2 = ['O', 'O']
x1 = [0.5, 0.8]
T = [298.15, 298.15]
# calculate x2 given x1
x2 = [1 - x for x in x1]

# create dataframes to hold the input data
test_df = pd.DataFrame({
    'Smiles_1': smiles_1,
    'Smiles_2': smiles_2,
})
features_df = pd.DataFrame({
    'x1': x1,
    'x2': x2,
    'T': T,
    'log10P1sat': ["nan"] * len(x1), # dummy values
    'log10P2sat': ["nan"] * len(x1), # dummy values
})

# store the input dataframes
test_df.to_csv(temp_test, index=False)
features_df.to_csv(temp_features, index=False)

In [36]:
# Preload the model object

arguments = [
    '--test_path', temp_test,
    '--features_path', temp_features,
    '--preds_path', temp_preds,
    '--checkpoint_path', model_path,
    '--number_of_molecules', '2',
    '--num_workers', '0',
]

args = PredictArgs().parse_args(arguments)

# loads the model specified in the args
# at this point, the model also sets the expected column titles for the test and features files.
# You can change the input data and features files later to do different predictions from the
# same model, but they have to have the same column titles the model expects. 
model_objects = load_model(args=args)

Loading training args
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "encoder.encoder.1.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.1.W_i.weight".
Loading pretrained parameter "encoder.encoder.1.W_h.weight".
Loading pretrained parameter "encoder.encoder.1.W_o.weight".
Loading pretrained parameter "encoder.encoder.1.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Loading pretrained parameter "intrinsic_vp.1.weight".
Loading pretrained parameter "intrinsic_vp.1.bias".
Loading pretrained parameter "intrinsic_vp.4.weight".
Load

In [37]:
# output: [y1,y2,log10P,lngamma1,lngamma2,log10P1sat,log10P2sat]
# makes the prediction specified by the test and features files from the args
preds = np.array(make_predictions(args=args, model_objects=model_objects)) 

Setting molecule featurization parameters to default.
Loading data


2it [00:00, 31418.01it/s]
100%|██████████| 2/2 [00:00<00:00, 23899.17it/s]


Validating SMILES
Test size = 2


100%|██████████| 1/1 [00:00<00:00, 68.83it/s]

Saving predictions to /tmp/tmpfseizuom.csv
Elapsed time = 0:00:00


In [38]:
# output: [y1,y2,log10P,lngamma1,lngamma2,log10P1sat,log10P2sat]
display(preds)

array([[ 6.66631699e-01,  3.33368212e-01,  3.84327793e+00,
         1.56077713e-01,  3.59869063e-01,  3.90041041e+00,
         3.51094317e+00],
       [ 9.95798230e-01,  4.20179963e-03,  5.03682470e+00,
        -1.32168308e-02, -3.49332243e-01,  5.13764620e+00,
         3.51094317e+00]])

In [41]:
# if you wanted to view it as a pandas dataframe
preds_df = pd.DataFrame(preds, columns=['y1', 'y2', 'log10P', 'lngamma1', 'lngamma2', 'log10P1sat', 'log10P2sat'])
display(preds_df)

,y1,y2,log10P,lngamma1,lngamma2,log10P1sat,log10P2sat
0,0.666632,0.333368,3.843278,0.156078,0.359869,3.900410,3.510943
1,0.995798,0.004202,5.036825,-0.013217,-0.349332,5.137646,3.510943


In [42]:
data_df = pd.read_csv("combined_vle.csv")

In [43]:
data_df

,SMILE 1,SMILE 2,y1,y2,log10P,x1,x2,T(K),log10P1sat,log10P2sat
0,CC#N,CCCCCl,0.8890,0.1110,5.005609,0.9620,0.0380,351.880,4.962623,5.031545
1,CC#N,CCCCCl,0.7760,0.2240,5.005609,0.9050,0.0950,349.070,4.923653,4.974931
2,CC#N,CCCCCl,0.7200,0.2800,5.005609,0.8660,0.1340,347.770,4.905385,4.957083
3,CC#N,CCCCCl,0.6670,0.3330,5.005609,0.8140,0.1860,346.570,4.888385,4.940467
4,CC#N,CCCCCl,0.6300,0.3700,5.005609,0.7710,0.2290,345.880,4.878551,4.930851
...,...,...,...,...,...,...,...,...,...,...
228823,CC(C)C(=O)O,CCCCCCC,0.1643,0.8357,3.860821,0.8261,0.1739,318.139,NaN,4.185089
228824,CC(C)C(=O)O,CCCCCCC,0.1864,0.8136,3.321356,0.8940,0.1060,298.144,NaN,3.776577
228825,CC(C)C(=O)O,CCCCCCC,0.2382,0.7618,3.728913,0.8940,0.1060,318.139,NaN,4.185089
228826,CC(C)C(=O)O,CCCCCCC,0.4558,0.5442,2.912366,0.9743,0.0257,298.144,NaN,3.776577


In [ ]:
# loop through all the data in the dataset one at a time
# grab out the relevant data for the specific row
# make them into input dataframes
# predict based on them using python make_predictions function
for i in range(10):
    row = data_df.iloc[i]
    s1 = row['SMILE 1']
    s2 = row['SMILE 2']
    x1 = row['x1']
    T = row['T(K)']

    test_df = pd.DataFrame({
        'Smiles_1': [s1],
        'Smiles_2': [s2],
    })
    features_df = pd.DataFrame({
        'x1': [x1],
        'x2': [x2],
        'T': [T],
        'log10P1sat': ["nan"], # dummy values
        'log10P2sat': ["nan"], # dummy values
    })

    print(s1, s2, x1, T)

CC#N CCCCCl 0.962 351.88
CC#N CCCCCl 0.905 349.07
CC#N CCCCCl 0.866 347.77
CC#N CCCCCl 0.814 346.57
CC#N CCCCCl 0.771 345.88
CC#N CCCCCl 0.726 345.32
CC#N CCCCCl 0.678 344.89
CC#N CCCCCl 0.616 344.51
CC#N CCCCCl 0.572 344.33
CC#N CCCCCl 0.52 344.22
